# week10_improved_week9_downloader.ipynb

This notebook is **self-contained** and will:
1) Install required libraries automatically.
2) Create a sample `image_urls.csv` if none exists.
3) Download images with retries, timeouts, and logging.
4) Save results to `download_log.csv` and `downloaded_images/`.
5) Render a thumbnail preview grid.

➡️ **Just run the notebook top to bottom. No edits needed.**

In [1]:
# Auto-install dependencies (VS Code Jupyter supported)
import sys, subprocess
pkgs = ['requests', 'pandas', 'pillow', 'urllib3', 'ipython']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '--disable-pip-version-check', *pkgs])
print('✅ Dependencies installed.')

✅ Dependencies installed.


In [2]:
# Imports & fixed parameters (no user edits required)
import os, re, time, hashlib, mimetypes
from pathlib import Path
from typing import Optional, Tuple
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from PIL import Image
from IPython.display import display, HTML

INPUT_CSV = 'image_urls.csv'          # auto-created below if not present
BASE_SAVE_DIR = Path('./')
SAVE_DIR = BASE_SAVE_DIR / 'downloaded_images'
LOG_PATH = BASE_SAVE_DIR / 'download_log.csv'

REQUEST_TIMEOUT = (8, 20)   # (connect, read) seconds
MAX_RETRIES = 3
BACKOFF_FACTOR = 0.8
RATE_LIMIT_SLEEP = 0.5
print('✅ Imports and parameters set.')

✅ Imports and parameters set.


In [3]:
# Create a demo CSV if none exists (no edits needed)
if not os.path.exists(INPUT_CSV):
    demo = pd.DataFrame({
        'url': [
            'https://upload.wikimedia.org/wikipedia/commons/3/3f/Fronalpstock_big.jpg',
            'https://upload.wikimedia.org/wikipedia/commons/4/47/PNG_transparency_demonstration_1.png'
        ],
        'filename': ['mountain_demo','transparency_demo'],
        'tag': ['demo','demo']
    })
    demo.to_csv(INPUT_CSV, index=False)
print(f'✅ CSV ready: {INPUT_CSV}')

✅ CSV ready: image_urls.csv


In [4]:
# Utility functions (safe CSV read with encoding fallbacks)
def read_csv_safely(path: str|Path) -> pd.DataFrame:
    path = str(path)
    if not os.path.exists(path):
        raise FileNotFoundError(f'CSV not found: {path}')
    for enc in ['utf-8', 'utf-8-sig', 'cp1252', 'latin1', 'ISO-8859-1']:
        try:
            df = pd.read_csv(path, encoding=enc)
            if not df.empty:
                return df
        except Exception:
            continue
    raise RuntimeError('Failed to read CSV with common encodings.')

def normalize_url(u: str) -> Optional[str]:
    if not isinstance(u, str):
        return None
    u = u.strip()
    if not u or not re.match(r'^https?://', u, flags=re.I):
        return None
    return u

def safe_stem(name: str) -> str:
    import re
    name = re.sub(r'[^A-Za-z0-9_.-]+', '_', str(name)).strip('_')
    return name or f'file_{int(time.time())}'
print('✅ Utility functions ready.')

✅ Utility functions ready.


In [5]:
# Networking helpers
def build_session() -> requests.Session:
    s = requests.Session()
    retries = Retry(total=MAX_RETRIES, backoff_factor=BACKOFF_FACTOR,
                    status_forcelist=[429, 500, 502, 503, 504],
                    allowed_methods=['GET'])
    s.headers.update({'User-Agent': 'Mozilla/5.0 (compatible; Week10-Downloader/1.0)'})
    s.mount('http://', HTTPAdapter(max_retries=retries))
    s.mount('https://', HTTPAdapter(max_retries=retries))
    return s

def guess_ext_from_response(resp: requests.Response, default: str='.jpg') -> str:
    ctype = resp.headers.get('Content-Type','')
    import mimetypes, os
    ext = mimetypes.guess_extension(ctype.split(';')[0].strip()) or ''
    if not ext:
        ext2 = os.path.splitext(resp.url.split('?')[0])[1]
        if ext2:
            return ext2
    return ext or default
print('✅ Networking configured.')

✅ Networking configured.


In [6]:
# Download one file
from typing import Tuple, Optional
def download_one(session: requests.Session, url: str, out_dir: Path, filename_stem: Optional[str]=None) -> Tuple[str, str, str]:
    out_dir.mkdir(parents=True, exist_ok=True)
    try:
        r = session.get(url, timeout=REQUEST_TIMEOUT, stream=True)
        r.raise_for_status()
        ext = guess_ext_from_response(r, default='.jpg')
        stem = safe_stem(filename_stem or os.path.splitext(os.path.basename(url.split('?')[0]))[0])
        out_path = out_dir / f"{stem}{ext}"
        if out_path.exists() and out_path.stat().st_size > 0:
            return url, str(out_path), 'skipped'
        with open(out_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
        time.sleep(RATE_LIMIT_SLEEP)
        # Validate image loosely
        try:
            Image.open(out_path).verify()
            status = 'ok'
        except Exception:
            status = 'ok_nonimage'
        return url, str(out_path), status
    except Exception as e:
        return url, str(out_dir), f'error: {e}'
print('✅ Download function ready.')

✅ Download function ready.


In [7]:
# Run the full pipeline: Load -> Clean -> Download -> Log
raw = read_csv_safely(INPUT_CSV)
if 'url' not in raw.columns:
    raise KeyError("CSV must contain a 'url' column")
raw['url_norm'] = raw['url'].map(normalize_url)
df = raw.dropna(subset=['url_norm']).drop_duplicates(subset=['url_norm']).copy()
df['filename'] = df['filename'] if 'filename' in df.columns else None
print(f"Loaded {len(raw)} rows; {len(df)} valid unique URLs.")

session = build_session()
records = []
for i, row in df.iterrows():
    url = row['url_norm']
    name = row['filename'] if isinstance(row.get('filename'), str) else None
    u, outp, status = download_one(session, url, SAVE_DIR, filename_stem=name)
    records.append({'url':u, 'saved_path':outp, 'status':status, 'tag': row.get('tag', None)})
    if (len(records) % 10)==0:
        print(f"Processed {len(records)}...")

log_df = pd.DataFrame.from_records(records)
log_df.to_csv(LOG_PATH, index=False)
display(log_df.head())
print(f"✅ Log saved to {LOG_PATH}")

Loaded 2 rows; 2 valid unique URLs.


,url,saved_path,status,tag
0,https://upload.wikimedia.org/wikipedia/commons...,downloaded_images/mountain_demo.jpg,ok,demo
1,https://upload.wikimedia.org/wikipedia/commons...,downloaded_images/transparency_demo.png,ok,demo


✅ Log saved to download_log.csv


In [8]:
# Thumbnail preview grid
def build_preview_grid(folder: Path, max_items: int = 24, thumb_h: int = 140) -> str:
    folder = Path(folder)
    items = []
    for p in sorted(folder.glob('*')):
        if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png','.gif','.webp','.bmp'}:
            items.append(p)
        if len(items) >= max_items:
            break
    html = ["<div style='display:flex;flex-wrap:wrap;gap:10px'>"]
    for p in items:
        html.append(f"<div style='text-align:center'><img src='{p.as_posix()}' style='height:{thumb_h}px'><br><span style='font-size:12px'>{p.name}</span></div>")
    html.append("</div>")
    return '\n'.join(html)

if SAVE_DIR.exists():
    display(HTML(build_preview_grid(SAVE_DIR)))
else:
    print('No images yet in', SAVE_DIR)
print('🎉 Done. You can now submit the notebook, the log CSV, and the images folder.')

🎉 Done. You can now submit the notebook, the log CSV, and the images folder.
